# 18 — Eksperimen dan evaluasi sistem RAG di Kaggle (gabungan 18 + 19)

**Notebook utama: generasi GGUF → tiga metrik RAGAS → rekap.**
Tidak membutuhkan folder aplikasi, server FastAPI, atau UI Sistem QA RAG. Model GGUF, dataset,
metadata dan indeks diambil dari Hugging Face seperti notebook 16; logika pipeline aplikasi
disertakan sebagai snapshot terbaca di bawah. Berkas aplikasi dan notebook 15–17 tidak diubah.

Urutan: Settings Kaggle → GPU + Internet ON → Add-ons/Secrets `HF_Faiss` (bila repo perlu token)
dan `OPENAI_API_KEY` → setup → konfigurasi → data/model → generasi → evaluasi → unduh output.
Default pilot **5 pertanyaan val**, 60 prediksi utama. OpenAI baru berjalan bila `RUN_RAGAS=True`.
Gunakan `DATA_SPLIT="test"`, `MAX_ITEMS=None`, dan RUN_NAME baru untuk final setelah konfigurasi dibekukan.
Jangan mengubah parameter berdasarkan skor test. Final 1.791 pertanyaan dengan dua protokol
berarti 21.492 prediksi, belum termasuk ekspansi, sitasi, dan panggilan judge: perlu beberapa sesi.

| Kondisi | Model | RAG |
|---|---|---|
| K1 | Instruct sebelum tuning | Tidak |
| K2 | Instruct sebelum tuning | Ya |
| K3 | Setelah QLoRA | Tidak |
| K4 | Setelah QLoRA | Ya |

Keduanya berlaku pada Gemma 2 dan Llama 3.2. `end_to_end` memakai model kondisi itu untuk
ekspansi kueri. `controlled` memakai dokumen tersimpan dari llama3.2-base yang sama bagi K2/K4,
agar perbandingan generator tidak tercampur perubahan retrieval. Hasil dua protokol dipisah.
K2−K1/K4−K3 membahas tambahan RAG; K3−K1 membahas tuning tanpa RAG; controlled K4−K2
membahas tuning dengan konteks tetap. RAGAS tidak menggunakan jawaban acuan cot + response.

Snapshot mengikuti algoritma aplikasi, tetapi CPU dipakai untuk embedding/reranker agar GPU
tersedia bagi Ollama; seed, num_ctx, timeout dan revisi HF dibuat eksplisit. **Kesetaraan metode,
bukan klaim waktu endpoint identik**. Uji aplikasi dan waktu server tetap dilakukan di notebook 20.
Prompt meminta penggunaan konteks bahkan pada tanpa-RAG (ablasi kebijakan prompt tetap);
K1/K3 dapat lebih sering menolak. Jangan menyimpulkannya sebagai kemampuan closed-book maksimum.

## 1. Setup paket (sesi Kaggle baru)

In [ ]:
import subprocess, sys
# Jalankan pada sesi Kaggle baru sebelum mengimpor pustaka lain.
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "numpy==1.26.4", "pandas==2.2.3", "requests>=2.32,<3",
        "transformers==4.46.3", "huggingface-hub==0.27.1",
        "langchain==0.3.14", "langchain-core==0.3.29", "datasets==3.2.0",
        "openai==1.59.7", "tiktoken==0.8.0",
        "sentence-transformers==3.3.1", "einops>=0.7,<1",
        "faiss-cpu==1.9.0.post1", "rank-bm25==0.2.2",
        "langchain-community==0.3.14", "langchain-openai==0.2.14",
        "ragas==0.2.15"])
# PyTorch/CUDA bawaan Kaggle tidak dipasang ulang. Jika diminta, restart kernel
# setelah instalasi lalu ubah INSTALL_DEPENDENCIES=False dan Run All kembali.

In [ ]:
from pathlib import Path
import os, json, hashlib, sys, platform, time, random
import importlib.metadata as metadata
import pandas as pd
import numpy as np

# Hitung identitas isi artefak untuk mencegah pencampuran eksperimen.
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

# Simpan JSON secara atomik agar berkas tidak setengah tertulis.
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)


# Ambil secret tanpa mencetak nilainya atau menyimpannya dalam artefak.
def read_secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            value = None
    if required and not value:
        raise RuntimeError(f"Aktifkan secret {name} pada Kaggle Add-ons > Secrets.")
    return value

HF_TOKEN = read_secret("HF_Faiss")
# OpenAI hanya dipanggil ketika RUN_RAGAS=True; key tidak ditulis ke notebook.

## 2. Konfigurasi dan pemulihan hasil

In [ ]:
RUN_NAME = "pilot_v2"
DATA_SPLIT = "val"      # Pilot/kalibrasi: val. Final: test, MAX_ITEMS=None, RUN_NAME baru.
MAX_ITEMS = 5
SEED = 42
RUN_EXPERIMENT = True
RUN_RAGAS = False       # Ubah True untuk mengizinkan panggilan OpenAI berbayar.
JUDGE_MODEL = "gpt-4o-mini-2024-07-18"
JUDGE_EMBEDDING = "text-embedding-3-small"
PROTOCOLS = ["end_to_end", "controlled"]
REFERENCE_MODEL = "llama3.2-base"
TOP_K = 5
NUM_CTX = 8192          # Konfigurasi eksplisit; pastikan semua input muat sebelum final.
RESTORE_DIR = ""        # Opsional: /kaggle/input/.../pilot_v2 (folder hasil run sebelumnya).
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / "rag_results" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
KB_DIR = WORK / "rag_assets" / "kb"
DATASET_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
DATASET_REVISION = "main"  # SHA tersimpan di manifest; resume otomatis memakai SHA lama.
KB_REVISION = "main"
INFERENCE_URL = "http://127.0.0.1:11434"
EMBEDDING_REVISION = "main"
RERANKER_REVISION = "main"

# Pulihkan artefak tanpa menimpa berkas berbeda pada tujuan.
def restore_run(source_dir, target_dir):
    import shutil
    src, dst = Path(source_dir).resolve(), Path(target_dir).resolve()
    if src == dst or src in dst.parents or dst in src.parents:
        raise ValueError("Folder sumber dan tujuan restore harus terpisah.")
    if not (src / "manifest.json").is_file():
        raise FileNotFoundError("RESTORE_DIR harus menunjuk folder yang berisi manifest.json.")
    files = [p for p in src.rglob("*") if p.is_file()]
    for p in files:
        target = dst / p.relative_to(src)
        if target.exists() and target.read_bytes() != p.read_bytes():
            raise ValueError(f"Konflik restore: {target.name}; pilih folder output kosong.")
    for p in files:
        target = dst / p.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            shutil.copy2(p, target)

if RESTORE_DIR:
    restore_run(RESTORE_DIR, OUT)
assert DATA_SPLIT in ["val", "test"]
assert DATA_SPLIT != "test" or MAX_ITEMS is None, "Pilot gunakan val; final test harus lengkap."
assert set(PROTOCOLS) <= {"end_to_end", "controlled"} and PROTOCOLS

## 3. Registrasi model dan prompt — snapshot konfigurasi aplikasi

In [ ]:
HF_KB_REPO = 'Makaareeem/publikasi-rag-knowledge-base'
EMBEDDING_MODEL_ID = 'nomic-ai/nomic-embed-text-v1.5'
QUERY_PREFIX = 'search_query: '
RETRIEVAL_K = 5
TOP_K_DENSE = 20
TOP_K_SPARSE = 20
TOP_K_RERANK_CANDIDATES = 20
RRF_K = 60
MAX_NEW_TOKENS = 512
RERANKER_MODEL_ID = 'madebyaris/rerank-indonesia'
QUERY_EXPANSION_TEMPERATURE = 0.3
QUERY_EXPANSION_MAX_TOKENS = 80
QUERY_EXPANSION_SYSTEM_PROMPT = 'Tugas Anda: tulis ulang SATU pertanyaan menjadi SATU kalimat pencarian yang lebih deskriptif untuk sistem pencarian dokumen statistik BPS. ATURAN KETAT — WAJIB DIIKUTI: 1. Output HANYA kalimat hasil tulis ulang. Tanpa kata pembuka, tanpa basa-basi, tanpa tanda kutip, tanpa penjelasan tambahan. 2. JANGAN menjawab pertanyaannya. JANGAN menyebutkan angka atau fakta spesifik apa pun. 3. JANGAN berkomentar tentang teks atau dokumen (dilarang keras memakai frasa seperti "teks menyatakan", "berdasarkan teks", "dokumen menyebutkan"). 4. JANGAN memulai dengan basa-basi asisten (dilarang keras memakai frasa seperti "tentu saja", "tentu", "baik", "saya dapat membantu"). 5. Maksimal 1 kalimat. Boleh tambahkan istilah/singkatan resmi BPS yang relevan bila ada.\n\nContoh:\nPertanyaan: TPAK berapa?\nHasil: Data Tingkat Partisipasi Angkatan Kerja (TPAK) penduduk Indonesia.\nPertanyaan: berapa kemiskinan 2024?\nHasil: Angka dan persentase penduduk miskin di Indonesia tahun 2024.'
CITATION_MAX_TOKENS = 500
CITATION_SYSTEM_PROMPT = 'Anda membantu merangkum jawaban singkat berdasarkan tiap sumber dokumen statistik BPS yang diberikan. Untuk SETIAP sumber bernomor, tulis satu kalimat yang menjawab pertanyaan HANYA berdasarkan isi sumber tersebut. Jika sumber itu tidak relevan/tidak menjawab pertanyaan, tulis "Sumber ini tidak membahas pertanyaan tersebut." Jangan mencampur informasi antar sumber. Jawab HANYA dalam format JSON: {"sources": [{"index": <nomor>, "answer": "<kalimat>"}, ...]}'
SYSTEM_PROMPT = 'Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS berdasarkan konteks yang diberikan. Kutip bagian konteks yang relevan sebelum menjawab, merujuk sumber HANYA dengan nomor sitasi seperti [1] [2], jangan menyebut ulang judul dokumen dalam kalimat jawaban karena berisiko salah mengaitkan dengan sumber lain. HANYA gunakan angka atau fakta yang secara eksplisit tertulis dalam konteks yang diberikan. JANGAN menambahkan angka dari pengetahuan Anda sendiri walau terasa familiar atau masuk akal. Jika konteks memuat TARGET, RENCANA, atau PROYEKSI (misalnya dari RPJMN atau sasaran kebijakan), jelaskan dengan jelas bahwa itu adalah target/rencana, BUKAN angka realisasi atau hasil pengukuran aktual, dan jangan menyamakan keduanya. Jika konteks yang diberikan tidak memuat informasi yang relevan dengan pertanyaan, katakan dengan jujur bahwa Anda tidak memiliki informasi tersebut dalam dokumen yang tersedia. Jangan mengarang jawaban atau menyimpulkan sesuatu yang tidak didukung oleh konteks.'
MODEL_REGISTRY = {'llama3.2-base': {'ollama_name': 'llama3.2-base', 'gguf_repo': 'Makaareeem/llama3.2-base-gguf', 'gguf_filename': 'Llama-3.2-3B-Instruct.Q4_K_M.gguf', 'system_role': True, 'repeat_penalty': 1.0}, 'llama3.2-finetuned': {'ollama_name': 'llama3.2-finetuned', 'gguf_repo': 'Makaareeem/llama3.2-finetuned-gguf', 'gguf_filename': 'Llama-3.2-3B-Instruct.Q4_K_M.gguf', 'system_role': True, 'repeat_penalty': 1.0}, 'gemma2-base': {'ollama_name': 'gemma2-base', 'gguf_repo': 'Makaareeem/gemma2-base-gguf', 'gguf_filename': 'gemma-2-2b-it.Q4_K_M.gguf', 'system_role': False, 'repeat_penalty': 1.0}, 'gemma2-finetuned': {'ollama_name': 'gemma2-finetuned', 'gguf_repo': 'Makaareeem/gemma2-finetuned-gguf', 'gguf_filename': 'gemma-2-2b-it.Q4_K_M.gguf', 'system_role': False, 'repeat_penalty': 1.0}}
DEFAULT_MODEL_KEY = 'llama3.2-finetuned'
RERANK_SCORE_THRESHOLD = -1.0

In [ ]:
PIPELINE_HASH = '31e573a7c9d5448ee96dc00b693b21a711c636075e2adfdc33968bf46acafc11'
APP_SOURCE_HASHES = {'citation_service.py': '0e056a723f003e5e39fcaac669fe7aac99a43fbfd91a6a491ca1f77bcefb020b', 'config.py': '8d7401fd97d8b85842ee57d6e223a9f1044341cbb7031cd0ec06fd7de6bcbfde', 'generation_service.py': '2f8bb8a281d74921c1d88c72a1c7110c47221748bce1142185481ad70f090130', 'main.py': '53d11aa0642d11a4f749c64e6b752941fc2bc05c68bada16f0cf7367db91288e', 'prompt_constructor.py': '22d2a6536cdb50b6dc47f2cc8001a84d95f790058a4c15b11af778ce686b02f5', 'query_expansion_service.py': '756f4ffb2d0f09aa7f9c10b07b5d32e20c5edb9ddc50b218da35744f1817db2a', 'reranker_service.py': 'c898c919666ec76a67d8e112d5be29e732aa71e9c5bedb4b74dbc4b13e13b540', 'retrieval_service.py': 'fb0ff55911a7db194a5cb864a8ec291570b95cb40e1c090f86389239f66c9f36', '__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}

## 4. Unduh dataset dan knowledge base dari HF

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from types import SimpleNamespace
import requests, subprocess, shutil
KB_CACHE_DIR = str(KB_DIR)
cfg = SimpleNamespace(**{k: v for k, v in list(globals().items()) if k.isupper() and k != "HF_TOKEN"})
api = HfApi(token=HF_TOKEN)
previous = json.loads((OUT / "manifest.json").read_text(encoding="utf-8")) if (OUT / "manifest.json").exists() else {}
saved = previous.get("asset_revisions", {})
ASSET_REVISIONS = {
    "dataset": api.repo_info(DATASET_REPO, repo_type="dataset", revision=saved.get("dataset", DATASET_REVISION)).sha,
    "kb": api.repo_info(HF_KB_REPO, repo_type="dataset", revision=saved.get("kb", KB_REVISION)).sha,
    "embedding": api.repo_info(EMBEDDING_MODEL_ID, revision=saved.get("embedding", EMBEDDING_REVISION)).sha,
    "reranker": api.repo_info(RERANKER_MODEL_ID, revision=saved.get("reranker", RERANKER_REVISION)).sha,
}
for key, model in MODEL_REGISTRY.items():
    ASSET_REVISIONS[key] = api.repo_info(model["gguf_repo"], revision=saved.get(key, "main")).sha
TEST_PATH = Path(hf_hub_download(DATASET_REPO, f"{DATA_SPLIT}.csv", repo_type="dataset",
    revision=ASSET_REVISIONS["dataset"], token=HF_TOKEN))
for name in ["chunk_metadata.jsonl", "faiss_index.bin"]:
    hf_hub_download(HF_KB_REPO, name, repo_type="dataset", revision=ASSET_REVISIONS["kb"],
                    token=HF_TOKEN, local_dir=KB_DIR)
print("Dataset:", DATA_SPLIT, "| revisi HF dibekukan (token tidak disimpan).")

In [ ]:
test = pd.read_csv(TEST_PATH, dtype=str, keep_default_na=False)
required = {"record_id", "instruction", "split"}
assert required <= set(test), required - set(test)
assert test.record_id.is_unique and test.record_id.str.strip().ne("").all()
assert test.instruction.str.strip().ne("").all()
assert test["split"].eq(DATA_SPLIT).all(), "Kolom split tidak sesuai DATA_SPLIT."
test = test.sort_values("record_id").reset_index(drop=True)
if MAX_ITEMS is not None:
    test = test.sample(n=min(MAX_ITEMS, len(test)), random_state=SEED).sort_values("record_id")
test = test.copy()
# Kolom acuan opsional hanya dipertahankan untuk pemeriksaan manusia di notebook 20.
if {"cot", "response"} <= set(test):
    test["ans_ref"] = test["cot"] + "\n\n" + test["response"]
selected_path = OUT / "test_selected.csv"
if selected_path.exists():
    prior = pd.read_csv(selected_path, dtype=str, keep_default_na=False)
    assert prior.to_dict("records") == test.to_dict("records"), "Sampel berubah; gunakan RUN_NAME baru."
else:
    test.to_csv(selected_path, index=False)
SCENARIOS = [
    {"architecture": arch, "condition": condition, "model_key": f"{arch}-{state}", "use_rag": rag}
    for arch in ["gemma2", "llama3.2"]
    for condition, state, rag in [("K1", "base", False), ("K2", "base", True),
                                  ("K3", "finetuned", False), ("K4", "finetuned", True)]
]
display(pd.DataFrame(SCENARIOS))
print("PILOT" if MAX_ITEMS is not None else "FULL TEST", len(test), "pertanyaan")

## 5. Fungsi pipeline aplikasi (self-contained)

Hybrid dense+BM25, RRF, reranker, ekspansi, prompt, dan sitasi. Tidak mengimpor `app`.

In [ ]:
import json
import faiss
from huggingface_hub import hf_hub_download
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer

class NomicQueryEmbeddings(Embeddings):

    # Komponen snapshot aplikasi: __init__.
    def __init__(self, model_id=EMBEDDING_MODEL_ID):
        self.model = SentenceTransformer(model_id, trust_remote_code=True, revision=ASSET_REVISIONS["embedding"], device="cpu")

    # Komponen snapshot aplikasi: embed_query.
    def embed_query(self, text):
        vector = self.model.encode(QUERY_PREFIX + text, normalize_embeddings=True)
        return vector.tolist()

    # Komponen snapshot aplikasi: embed_documents.
    def embed_documents(self, texts):
        raise NotImplementedError('Dokumen sudah diembed pada NB07; tidak dipakai ulang di sini.')

# Komponen snapshot aplikasi: download_kb_artifacts.
def download_kb_artifacts():
    index_path = hf_hub_download(HF_KB_REPO, 'faiss_index.bin', repo_type='dataset', token=HF_TOKEN, local_dir=KB_CACHE_DIR, revision=ASSET_REVISIONS["kb"])
    metadata_path = hf_hub_download(HF_KB_REPO, 'chunk_metadata.jsonl', repo_type='dataset', token=HF_TOKEN, local_dir=KB_CACHE_DIR, revision=ASSET_REVISIONS["kb"])
    return (index_path, metadata_path)

# Komponen snapshot aplikasi: load_documents.
def load_documents(metadata_path):
    documents = []
    with open(metadata_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            documents.append(Document(page_content=record['text'], metadata=record))
    return documents

# Komponen snapshot aplikasi: adapt_faiss_to_langchain.
def adapt_faiss_to_langchain(index_path, documents, embeddings):
    raw_index = faiss.read_index(index_path)
    docstore_dict = {}
    index_to_docstore_id = {}
    for doc in documents:
        faiss_id = doc.metadata['faiss_id']
        doc_id = str(faiss_id)
        docstore_dict[doc_id] = doc
        index_to_docstore_id[faiss_id] = doc_id
    return FAISS(embedding_function=embeddings, index=raw_index, docstore=InMemoryDocstore(docstore_dict), index_to_docstore_id=index_to_docstore_id)

# Komponen snapshot aplikasi: build_indices.
def build_indices():
    index_path, metadata_path = download_kb_artifacts()
    documents = load_documents(metadata_path)
    embeddings = NomicQueryEmbeddings()
    vectorstore = adapt_faiss_to_langchain(index_path, documents, embeddings)
    bm25_retriever = BM25Retriever.from_documents(documents, preprocess_func=lambda text: text.lower().split())
    bm25_retriever.k = TOP_K_SPARSE
    return (vectorstore, bm25_retriever)

# Komponen snapshot aplikasi: reciprocal_rank_fusion.
def reciprocal_rank_fusion(rankings, k=RRF_K):
    scores = {}
    doc_lookup = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            doc_id = doc.metadata['faiss_id']
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            doc_lookup[doc_id] = doc
    ranked_ids = sorted(scores, key=scores.get, reverse=True)
    return [doc_lookup[doc_id] for doc_id in ranked_ids]

# Komponen snapshot aplikasi: hybrid_search.
def hybrid_search(vectorstore, bm25_retriever, queries, dense_k=TOP_K_DENSE, sparse_k=TOP_K_SPARSE):
    rankings = []
    for query in queries:
        dense_docs = [doc for doc, _ in vectorstore.similarity_search_with_score(query, k=dense_k)]
        rankings.append(dense_docs)
        sparse_docs = bm25_retriever.invoke(query)[:sparse_k]
        rankings.append(sparse_docs)
    fused = reciprocal_rank_fusion(rankings)
    return fused[:TOP_K_RERANK_CANDIDATES]

# Komponen snapshot aplikasi: format_source.
def format_source(doc, score):
    m = doc.metadata
    return {'chunk_id': m.get('chunk_id'), 'document_title': m.get('document_title'), 'section_id': m.get('section_id'), 'page_start': m.get('page_start'), 'page_end': m.get('page_end'), 'bps_url': m.get('bps_url'), 'score': float(score)}

from sentence_transformers import CrossEncoder
_model = None

# Komponen snapshot aplikasi: get_model.
def get_model():
    global _model
    if _model is None:
        _model = CrossEncoder(RERANKER_MODEL_ID, revision=ASSET_REVISIONS["reranker"], device="cpu")
    return _model

# Komponen snapshot aplikasi: rerank.
def rerank(question, docs, top_k=RETRIEVAL_K):
    if not docs:
        return ([], [])
    model = get_model()
    pairs = [[question, doc.page_content] for doc in docs]
    scores = model.predict(pairs)
    ranked = sorted([(doc, float(score)) for doc, score in zip(docs, scores)], key=lambda pair: pair[1], reverse=True)
    passed = [(doc, score) for doc, score in ranked if score >= RERANK_SCORE_THRESHOLD]
    return (passed[:top_k], ranked)

import json
import requests

# Komponen snapshot aplikasi: generate.
def generate(model_name, messages, repeat_penalty=1.0, temperature=0, num_predict=MAX_NEW_TOKENS, response_format=None):
    payload = {'model': model_name, 'messages': messages, 'stream': False, 'options': {'temperature': temperature, 'num_predict': num_predict, 'repeat_penalty': repeat_penalty, 'seed': SEED, 'num_ctx': NUM_CTX}}
    if response_format:
        payload['format'] = response_format
    resp = requests.post(f'{INFERENCE_URL}/api/chat', json=payload, timeout=(10, 900))
    resp.raise_for_status()
    return resp.json()['message']['content']

# Komponen snapshot aplikasi: parse_json_response.
def parse_json_response(raw, caller):
    text = raw.strip()
    if text.startswith('```'):
        text = text.strip('`')
        if text.lower().startswith('json'):
            text = text[4:]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print(f'[{caller}] Gagal parse JSON dari model, output mentah:\n{raw}\n')
        return None

# Komponen snapshot aplikasi: health.
def health():
    try:
        requests.get(f'{INFERENCE_URL}/api/tags', timeout=5).raise_for_status()
        return True
    except requests.RequestException:
        return False

# Komponen snapshot aplikasi: format_context.
def format_context(docs):
    blocks = []
    for i, doc in enumerate(docs, 1):
        header = f'[{i}] {doc.metadata.get('document_title', '-')}'
        if doc.metadata.get('section_id'):
            header += f' | {doc.metadata['section_id']}'
        blocks.append(f'{header}\n{doc.page_content}')
    return '\n\n'.join(blocks)

# Komponen snapshot aplikasi: build_user_content.
def build_user_content(question, docs, use_rag):
    if use_rag and (not docs):
        return f'Pertanyaan: {question}\n\n(Tidak ditemukan bagian dokumen yang cukup relevan dengan pertanyaan ini.)'
    if not docs:
        return f'Pertanyaan: {question}'
    return f'Konteks:\n{format_context(docs)}\n\nPertanyaan: {question}'

# Komponen snapshot aplikasi: build_messages.
def build_messages(question, docs, system_role: bool, use_rag: bool=True):
    user_content = build_user_content(question, docs, use_rag)
    if system_role:
        return [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_content}]
    return [{'role': 'user', 'content': f'{SYSTEM_PROMPT}\n\n{user_content}'}]

import re
_PREAMBLE_RE = re.compile('^(tentu(\\s+saja)?|baik|oke|ok|teks (ini |tersebut )?menyatakan|berdasarkan (teks|dokumen)|dokumen (ini |tersebut )?menyebutkan|saya (dapat|bisa) membantu)[^\\n.:]*[.:,]\\s*', re.IGNORECASE)

# Komponen snapshot aplikasi: _clean_expansion.
def _clean_expansion(text):
    cleaned = text.strip().strip('"\'“”')
    prev = None
    while prev != cleaned:
        prev = cleaned
        cleaned = _PREAMBLE_RE.sub('', cleaned).strip().strip('"\'“”')
    return cleaned

# Komponen snapshot aplikasi: expand_query.
def expand_query(question, model_cfg):
    if model_cfg['system_role']:
        messages = [{'role': 'system', 'content': QUERY_EXPANSION_SYSTEM_PROMPT}, {'role': 'user', 'content': question}]
    else:
        messages = [{'role': 'user', 'content': f'{QUERY_EXPANSION_SYSTEM_PROMPT}\n\n{question}'}]
    raw = generate(model_cfg['ollama_name'], messages, repeat_penalty=model_cfg['repeat_penalty'], temperature=QUERY_EXPANSION_TEMPERATURE, num_predict=QUERY_EXPANSION_MAX_TOKENS)
    expanded = _clean_expansion(raw)
    return [expanded] if expanded else []

# Komponen snapshot aplikasi: generate_citations.
def generate_citations(question, docs, model_cfg):
    if not docs:
        return {}
    sources_block = '\n\n'.join((f'[{i}] {doc.page_content}' for i, doc in enumerate(docs, 1)))
    user_content = f'Pertanyaan: {question}\n\n{sources_block}'
    if model_cfg['system_role']:
        messages = [{'role': 'system', 'content': CITATION_SYSTEM_PROMPT}, {'role': 'user', 'content': user_content}]
    else:
        messages = [{'role': 'user', 'content': f'{CITATION_SYSTEM_PROMPT}\n\n{user_content}'}]
    raw = generate(model_cfg['ollama_name'], messages, repeat_penalty=model_cfg['repeat_penalty'], temperature=0, num_predict=CITATION_MAX_TOKENS, response_format='json')
    data = parse_json_response(raw, 'citation')
    if not isinstance(data, dict):
        return {}
    result = {}
    for item in data.get('sources', []):
        if not isinstance(item, dict) or 'index' not in item or 'answer' not in item:
            continue
        try:
            result[int(item['index'])] = item['answer']
        except (ValueError, TypeError):
            continue
    return result

In [ ]:
# Hash berkas besar secara bertahap tanpa memuatnya seluruhnya ke RAM.
def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Catat versi pustaka yang benar-benar terpasang.
def package_versions():
    names = ["pandas", "numpy", "requests", "torch", "transformers", "sentence-transformers",
             "langchain-core", "langchain-community", "faiss-cpu", "rank-bm25"]
    return {n: metadata.version(n) for n in names}

In [ ]:
# Siapkan runtime Kaggle seperti NB16; tidak membutuhkan aplikasi FastAPI/UI.
def ensure_kaggle_ollama():
    if not shutil.which("ollama"):
        if not Path("/kaggle/working").exists():
            raise RuntimeError("Di luar Kaggle, pasang dan jalankan Ollama secara manual.")
        subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
        installer = WORK / "install_ollama.sh"
        response = requests.get("https://ollama.com/install.sh", timeout=60)
        response.raise_for_status()
        installer.write_text(response.text, encoding="utf-8")
        subprocess.run(["sh", str(installer)], check=True)
    try:
        requests.get(INFERENCE_URL + "/api/version", timeout=5).raise_for_status()
        return
    except requests.RequestException:
        if not Path("/kaggle/working").exists():
            raise RuntimeError("Jalankan ollama serve terlebih dahulu.") from None
    env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MAX_LOADED_MODELS="1")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            requests.get(INFERENCE_URL + "/api/version", timeout=2).raise_for_status()
            return
        except requests.RequestException:
            time.sleep(1)
    raise RuntimeError("Ollama belum siap; periksa log sesi Kaggle.")

# Unduh empat GGUF dari repo yang sama dengan NB16 dan registrasikan pada Ollama.
def prepare_models():
    info = {}
    for key, model in MODEL_REGISTRY.items():
        path = Path(hf_hub_download(model["gguf_repo"], model["gguf_filename"],
                                   revision=ASSET_REVISIONS[key], token=HF_TOKEN))
        lines = [f'FROM "{path.as_posix()}"']
        if model["system_role"]:
            lines.append('SYSTEM """' + SYSTEM_PROMPT + '"""')
        lines += ["PARAMETER temperature 0", f"PARAMETER num_predict {MAX_NEW_TOKENS}",
                  f"PARAMETER repeat_penalty {model['repeat_penalty']}", f"PARAMETER num_ctx {NUM_CTX}"]
        modelfile = WORK / "rag_assets" / f"{key}.Modelfile"
        modelfile.parent.mkdir(parents=True, exist_ok=True)
        modelfile.write_text("\n".join(lines), encoding="utf-8")
        subprocess.run(["ollama", "create", model["ollama_name"], "-f", str(modelfile)], check=True)
        info[key] = {"repo": model["gguf_repo"], "filename": path.name,
                     "revision": ASSET_REVISIONS[key], "sha256": file_hash(path)}
    return info

In [ ]:
# Bentuk identitas evaluasi unik meskipun chunk_id asal berulang pada KB.
def evaluation_chunk_id(record):
    return f"{record['chunk_id']}::faiss={record['faiss_id']}"

# Muat metadata lokal menggunakan gabungan chunk_id dan faiss_id.
def load_chunks(path):
    records = [json.loads(s) for s in Path(path).read_text(encoding="utf-8").splitlines() if s.strip()]
    chunks = {evaluation_chunk_id(r): r for r in records}
    assert len(chunks) == len(records), "Identitas gabungan metadata duplikat."
    assert len({r["faiss_id"] for r in records}) == len(records), "faiss_id harus unik."
    return chunks



## 7. Generasi dan resume

Hash manifest mencegah pencampuran konfigurasi. Hasil sukses dilewati; error dicoba ulang.
Simpan semua output Kaggle, attach kembali sebagai input, isi RESTORE_DIR untuk sesi berikutnya.
Jika versi Ollama/paket berubah, pulihkan versi yang dicatat atau gunakan run baru.
`num_ctx` bukan bukti tidak ada pemotongan: audit panjang prompt sebelum final; jangan klaim truncation telah diukur.

In [ ]:
# Bekukan isi eksperimen; catat perangkat terpisah agar resume sesi baru tetap mungkin.
def freeze_manifest():
    runtime = requests.get(INFERENCE_URL + "/api/version", timeout=15)
    runtime.raise_for_status()
    settings = {k: getattr(cfg, k) for k in ["EMBEDDING_MODEL_ID", "QUERY_PREFIX", "RERANKER_MODEL_ID",
        "RERANK_SCORE_THRESHOLD", "TOP_K_DENSE", "TOP_K_SPARSE", "RRF_K", "TOP_K_RERANK_CANDIDATES",
        "MAX_NEW_TOKENS", "QUERY_EXPANSION_TEMPERATURE", "QUERY_EXPANSION_MAX_TOKENS", "CITATION_MAX_TOKENS"]}
    manifest = {"seed": SEED, "pilot": MAX_ITEMS is not None, "split": DATA_SPLIT,
        "protocols": PROTOCOLS, "reference_model": REFERENCE_MODEL, "top_k": TOP_K,
        "scenarios": SCENARIOS, "test_hash": file_hash(TEST_PATH), "selected_hash": digest(test.to_dict("records")),
        "asset_revisions": ASSET_REVISIONS, "models": MODEL_ARTIFACTS, "settings": settings,
        "num_ctx": NUM_CTX, "pipeline_hash": PIPELINE_HASH, "app_source_hashes": APP_SOURCE_HASHES,
        "prompts_hash": digest([SYSTEM_PROMPT, QUERY_EXPANSION_SYSTEM_PROMPT, CITATION_SYSTEM_PROMPT]),
        "kb_hashes": {n: file_hash(KB_DIR / n) for n in ["chunk_metadata.jsonl", "faiss_index.bin"]},
        "versions": package_versions(), "ollama": runtime.json()}
    manifest["experiment_id"] = digest(manifest)
    path = OUT / "manifest.json"
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != manifest:
        raise ValueError("Konfigurasi/runtime berubah. Gunakan RUN_NAME baru atau pulihkan versi lama.")
    write_json(path, manifest)
    write_json(OUT / "environments" / (time.strftime("%Y%m%d_%H%M%S") + ".json"),
        {"platform": platform.platform(), "python": sys.version, "processor": platform.processor()})
    return manifest

In [ ]:
MODEL_ARTIFACTS = {}
if RUN_EXPERIMENT:
    ensure_kaggle_ollama()
    MODEL_ARTIFACTS = prepare_models()

In [ ]:
# Jalankan ekspansi, hybrid search dan reranking yang sama dengan aplikasi.
def retrieve(question, model_key, vectorstore, bm25):
    start = time.perf_counter()
    expansions = expand_query(question, cfg.MODEL_REGISTRY[model_key])
    candidates = hybrid_search(vectorstore, bm25, [question] + expansions)
    scored, ranked = rerank(question, candidates, top_k=TOP_K)
    return {"expansions": expansions, "retrieval_s": time.perf_counter() - start,
            "docs": [{"text": d.page_content, "metadata": d.metadata, "score": float(s)} for d, s in scored],
            "ranked_ids": [evaluation_chunk_id(d.metadata) for d, _ in ranked],
            "candidate_ids": [evaluation_chunk_id(d.metadata) for d in candidates]}

# Kembalikan dokumen cache ke objek yang diterima penyusun prompt aplikasi.
def restore_docs(snapshot):
    from langchain_core.documents import Document
    return [Document(page_content=d["text"], metadata=d["metadata"]) for d in snapshot["docs"]]

# Simpan satu hasil lengkap segera setelah selesai agar proses dapat dilanjutkan.
def run_one(ex, scenario, protocol, manifest, vectorstore, bm25):
    key = f"{ex['record_id']}__{scenario['model_key']}__{scenario['condition']}__{protocol}"
    path = OUT / "predictions" / (key + ".json")
    if path.exists():
        old = json.loads(path.read_text(encoding="utf-8"))
        assert old["experiment_id"] == manifest["experiment_id"]
        if old["status"] == "ok":
            return old
    row = {**scenario, "record_id": ex["record_id"], "protocol": protocol,
           "question": ex["instruction"], "ans_ref": ex.get("ans_ref", ""),
           "subtask_type": ex.get("subtask_type", ""), "experiment_id": manifest["experiment_id"]}
    start = time.perf_counter()
    try:
        snap = {"docs": [], "expansions": [], "retrieval_s": 0., "ranked_ids": [], "candidate_ids": []}
        cache_hit = False
        if scenario["use_rag"]:
            if protocol == "controlled":
                cache = OUT / "retrieval_cache" / (ex["record_id"] + ".json")
                if cache.exists():
                    snap = json.loads(cache.read_text(encoding="utf-8"))
                    assert snap["experiment_id"] == manifest["experiment_id"]
                    cache_hit = True
                else:
                    snap = retrieve(ex["instruction"], REFERENCE_MODEL, vectorstore, bm25)
                    snap["experiment_id"] = manifest["experiment_id"]
                    write_json(cache, snap)
            else:
                snap = retrieve(ex["instruction"], scenario["model_key"], vectorstore, bm25)
        docs = restore_docs(snap)
        model_cfg = cfg.MODEL_REGISTRY[scenario["model_key"]]
        t = time.perf_counter()
        citations = generate_citations(ex["instruction"], docs, model_cfg)
        citation_s = time.perf_counter() - t
        messages = build_messages(ex["instruction"], docs, model_cfg["system_role"], scenario["use_rag"])
        t = time.perf_counter()
        answer = generate(model_cfg["ollama_name"], messages, repeat_penalty=1.0, temperature=0, num_predict=512)
        generation_s = time.perf_counter() - t
        # Tiap blok konteks persis mengikuti header dan teks yang masuk prompt.
        context_text = format_context(docs)
        row.update(status="ok", answer=answer, messages=messages,
                   contexts=[context_text] if docs else [],
                   context_chunks=[d.page_content for d in docs],
                   retrieved_ids=[evaluation_chunk_id(d.metadata) for d in docs],
                   source_chunk_ids=[d.metadata["chunk_id"] for d in docs],
                   retrieval=snap, context_hash=digest(context_text), citations=citations,
                   retrieval_cache_hit=cache_hit, retrieval_s=snap["retrieval_s"],
                   citation_s=citation_s, generation_s=generation_s,
                   total_s=time.perf_counter() - start, output_chars=len(answer))
    except Exception as exc:
        row.update(status="error", error=f"{type(exc).__name__}: {exc}", total_s=time.perf_counter()-start)
    write_json(path, row)
    return row

# Jalankan kondisi dalam urutan acak tetap; simpan kegagalan, bukan jawaban buatan.
def run_experiments():
    manifest = freeze_manifest()
    vectorstore, bm25 = build_indices()
    # Bekukan seluruh retrieval controlled sebelum pergantian model generator.
    if 'controlled' in PROTOCOLS:
        for ex in test.to_dict('records'):
            cache = OUT / 'retrieval_cache' / (ex['record_id'] + '.json')
            if cache.exists():
                assert json.loads(cache.read_text(encoding='utf-8'))['experiment_id'] == manifest['experiment_id']
                continue
            snapshot = retrieve(ex['instruction'], REFERENCE_MODEL, vectorstore, bm25)
            snapshot['experiment_id'] = manifest['experiment_id']
            write_json(cache, snapshot)
    jobs = [(ex, s, p) for ex in test.to_dict("records") for s in SCENARIOS for p in PROTOCOLS
            if p != "controlled" or s["use_rag"]]
    random.Random(SEED).shuffle(jobs)
    jobs.sort(key=lambda job: job[1]["model_key"])  # Kurangi bongkar-muat GPU.
    records = []
    for i, (ex, s, p) in enumerate(jobs, 1):
        result = run_one(ex, s, p, manifest, vectorstore, bm25)
        records.append(result)
        print(f"{i}/{len(jobs)} {s['model_key']} {s['condition']} {p}: {result['status']}")
    return records

if RUN_EXPERIMENT:
    records = run_experiments()
    # Lepas GGUF setelah generasi selesai; penilaian RAGAS memakai API.
    for model in MODEL_REGISTRY.values():
        response = requests.post(INFERENCE_URL + '/api/generate',
            json={'model': model['ollama_name'], 'keep_alive': 0}, timeout=60)
        response.raise_for_status()
else:
    print("Konfigurasi siap; RUN_EXPERIMENT=False. Belum ada inferensi yang dijalankan.")

## 8. Evaluasi RAGAS saja (tidak perlu menjalankan notebook 19)

Tanpa RAG: answer relevance. Dengan RAG: answer relevance, faithfulness, context relevance.
Metrik berbasis jawaban acuan tetap berada pada notebook tahap sebelumnya, tidak dihitung di sini.
Tidak membutuhkan qrels. Faithfulness menilai dukungan jawaban, bukan retrieval secara langsung.

In [ ]:
# Baca prediksi dan pastikan hasil berasal dari satu eksperimen yang sama.
def load_predictions():
    manifest = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((OUT / "predictions").glob("*.json"))]
    if not rows:
        raise ValueError("Jalankan notebook 18 terlebih dahulu.")
    assert all(r["experiment_id"] == manifest["experiment_id"] for r in rows)
    df = pd.DataFrame(rows)
    assert not df.duplicated(["record_id", "model_key", "condition", "protocol"]).any()
    return df, manifest

predictions, manifest = load_predictions()
display(predictions.groupby(["protocol", "model_key", "condition", "status"]).size().rename("n").reset_index())
# Siapkan masukan RAGAS tanpa jawaban acuan, qrels, atau metrik tahap sebelumnya.
def prepare_ragas_inputs(df):
    required = {"record_id", "model_key", "architecture", "condition", "protocol", "status"}
    if not required <= set(df):
        raise ValueError("Identitas/status prediksi tidak lengkap.")
    scored = df[df.status.eq("ok")].copy().reset_index(drop=True)
    if scored.empty:
        raise ValueError("Tidak ada prediksi berhasil; periksa inference_coverage.csv.")
    if not {"question", "answer", "contexts", "use_rag", "context_hash"} <= set(scored):
        raise ValueError("Masukan RAGAS tidak lengkap.")
    assert scored.question.map(lambda x: isinstance(x, str) and bool(x.strip())).all()
    assert scored.answer.map(lambda x: isinstance(x, str)).all()
    assert scored.contexts.map(lambda x: isinstance(x, list) and all(isinstance(y, str) for y in x)).all()
    assert scored.use_rag.map(lambda x: isinstance(x, (bool, np.bool_))).all()
    assert scored.loc[~scored.use_rag, "contexts"].map(len).eq(0).all(), "Tanpa RAG harus tanpa konteks."
    # Allowlist memastikan skor/kolom acuan lama tidak masuk ekspor baru.
    columns = ["record_id", "model_key", "architecture", "condition", "protocol", "status",
               "question", "answer", "contexts", "use_rag", "context_hash", "experiment_id", "subtask_type"]
    scored = scored[[c for c in columns if c in scored]].copy()
    if "subtask_type" not in scored:
        scored["subtask_type"] = ""
    return scored

predictions.groupby(["protocol", "architecture", "condition", "status"]).size().rename("n").to_csv(
    OUT / "inference_coverage.csv")
scored = prepare_ragas_inputs(predictions)


## RAGAS — tiga penilaian, OpenAI sebagai judge

1. **Faithfulness**: apakah klaim jawaban didukung konteks retrieval yang masuk prompt?
2. **Answer relevance** (`answer_relevancy` / `ResponseRelevancy`): kesesuaian jawaban dengan pertanyaan,
   memakai pertanyaan hasil rekonstruksi dan embedding OpenAI; strictness=3.
3. **Context relevance** (`context_relevance` / `ContextRelevance`): kesesuaian konteks dengan pertanyaan.

RAGAS 0.2.15 menyediakan ContextRelevance pada keluarga metrik NVIDIA: dua penilaian 0/1/2,
dinormalisasi menjadi 0–1 dan dirata-ratakan. **Ini bukan rumus fraksi kalimat relevan pada paper
RAGAS awal, bukan context precision, dan bukan context recall.** Definisi operasional ini harus
ditulis konsisten di Bab III. Tidak membutuhkan qrels atau kolom recall pada dataset.
Audit kode wheel 0.2.15 menemukan batas internal 7.000 karakter konteks, temperature=0.1,
dan template pertama dipanggil dua kali (berbeda dari uraian dokumentasi). Implementasi paket
dipertahankan; context_relevance_truncated mencatat pemotongan. Jangan mengklaim seluruh
konteks panjang dinilai. Audit dampak batas ini sebelum final.

Sumber: [faithfulness](https://docs.ragas.io/en/v0.2.15/concepts/metrics/available_metrics/faithfulness/),
[answer relevance](https://docs.ragas.io/en/v0.2.15/concepts/metrics/available_metrics/answer_relevance/),
[context relevance](https://docs.ragas.io/en/v0.2.15/concepts/metrics/available_metrics/nvidia_metrics/).

Aktifkan secret `OPENAI_API_KEY`, lalu `RUN_RAGAS=True`. Pertanyaan, jawaban dan potongan dokumen
dikirim ke OpenAI untuk penilaian; model yang diuji tetap GGUF lokal Kaggle, bukan OpenAI.
Ada biaya chat + embedding, dan satu metrik dapat memanggil API beberapa kali. Mulai dari pilot.
RAGAS tidak melatih ulang model. Nilai error/NaN dan empty retrieval dilaporkan bersama cakupan,
bukan diubah menjadi nol atau dianggap berhasil. Faithfulness/context relevance tidak berlaku pada
K1/K3; answer relevance tetap dihitung. Audit kecakapan judge pada bahasa Indonesia secara manual.

In [ ]:
# Susun evaluator OpenAI eksplisit; key hanya berada di memori proses.
def make_ragas_evaluators():
    assert metadata.version("ragas") == "0.2.15", "Gunakan versi paket yang dibekukan."
    from ragas.metrics import Faithfulness, ResponseRelevancy, ContextRelevance
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.run_config import RunConfig
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    key = read_secret("OPENAI_API_KEY", required=True)
    judge = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, api_key=key, temperature=0,
        max_tokens=4096, timeout=120, max_retries=2), run_config=RunConfig(timeout=180, max_retries=2))
    embedding = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model=JUDGE_EMBEDDING, api_key=key,
        request_timeout=120, max_retries=2))
    return {"faithfulness": Faithfulness(llm=judge),
            "answer_relevancy": ResponseRelevancy(llm=judge, embeddings=embedding, strictness=3),
            "context_relevance": ContextRelevance(llm=judge)}

# Nilai tiap metrik dengan cache terpisah; kegagalan tidak mengulang metrik yang berhasil.
async def score_ragas_cached(df, metrics, config, sample_factory):
    import asyncio
    evaluator_id = digest(config)
    write_json(OUT / "ragas_evaluators" / (evaluator_id + ".json"), config)
    results = []
    for row in df.to_dict("records"):
        context_chars = len('\n'.join(row['contexts']))
        result = {"ragas_evaluator_id": evaluator_id,
                  "context_relevance_context_chars": context_chars,
                  "context_relevance_truncated": bool(row['use_rag'] and context_chars > 7000)}
        for name, metric in metrics.items():
            if name != "answer_relevancy" and (not row["use_rag"] or not row["contexts"]):
                result[name] = np.nan
                result[name + "_status"] = "no_rag_not_applicable" if not row["use_rag"] else "empty_retrieval"
                continue
            inputs = {"user_input": row["question"]}
            if name != "context_relevance":
                inputs["response"] = row["answer"]
            if name != "answer_relevancy":
                inputs["retrieved_contexts"] = row["contexts"]
            cache_key = digest({"evaluator": evaluator_id, "metric": name, "inputs": inputs})
            path = OUT / "ragas_cache" / (cache_key + ".json")
            cached = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
            if cached.get("status") != "ok":
                try:
                    value = float(await asyncio.wait_for(metric.single_turn_ascore(sample_factory(**inputs)), timeout=600))
                    cached = {"value": value if np.isfinite(value) else None,
                              "status": "ok" if np.isfinite(value) else "judge_nan"}
                except Exception as exc:
                    # Jangan simpan raw exception provider yang dapat memuat rincian kredensial.
                    cached = {"value": None, "status": "error_" + type(exc).__name__}
                write_json(path, cached)
            result[name] = np.nan if cached["value"] is None else cached["value"]
            result[name + "_status"] = cached["status"]
        results.append(result)
    return pd.DataFrame(results)

# Jalankan tiga metrik RAGAS terhadap prompt context aktual, bukan konteks jawaban acuan.
async def evaluate_ragas_rows(df):
    from ragas.dataset_schema import SingleTurnSample
    metrics = make_ragas_evaluators()
    config = {"implementation": "openai_three_metrics_v1", "judge": JUDGE_MODEL,
        "embedding": JUDGE_EMBEDDING, "configured_chat_temperature": 0, "max_tokens": 4096,
        "context_relevance_internal_temperature": 0.1, "context_relevance_max_chars": 7000,
        "context_relevance_prompts": "0.2.15 wheel repeats template_relevance1 twice",
        "strictness": 3, "prompt_language": "English default; audit Indonesian quality",
        "context_relevance_definition": "ragas NVIDIA ContextRelevance: two 0/1/2 judgments normalized",
        "versions": {n: metadata.version(n) for n in ["ragas", "langchain-openai", "openai"]}}
    return await score_ragas_cached(df, metrics, config, SingleTurnSample)

if RUN_RAGAS:
    ragas_scores = await evaluate_ragas_rows(scored)
    scored = pd.concat([scored.reset_index(drop=True), ragas_scores], axis=1)
else:
    for metric in ["faithfulness", "answer_relevancy", "context_relevance"]:
        scored[metric] = np.nan
        scored[metric + "_status"] = "not_run"
    print("RAGAS belum dijalankan. Aktifkan RUN_RAGAS=True setelah secret dan biaya disetujui.")

## Ringkasan RAGAS, kelengkapan, dan selisih berpasangan

Rata-rata tidak cukup untuk menyimpulkan pengaruh komponen: laporkan selisih pada record_id yang sama,
jumlah pasangan lengkap, kegagalan, serta distribusi menurut jenis tugas. Perbandingan end-to-end K4–K2
mengandung perubahan ekspansi kueri; perbandingan controlled K4–K2 menggunakan konteks identik.
Ringkasan ini deskriptif, tidak menganggap pertanyaan dari bagian dokumen yang sama saling independen.

In [ ]:
METRICS = ["faithfulness", "answer_relevancy", "context_relevance"]
GROUP = ["protocol", "architecture", "condition"]

# Ringkas setiap metrik beserta penyebut valid, simpangan baku, dan median.
def summarize(df, groups):
    return df.groupby(groups, dropna=False)[METRICS].agg(["mean", "count", "std", "median"])

# Hitung selisih kondisi hanya pada pertanyaan yang sama-sama berhasil dinilai.
def paired_differences(df):
    rows = []
    for (protocol, arch), group in df.groupby(["protocol", "architecture"]):
        contrasts = [("K2", "K1"), ("K3", "K1"), ("K4", "K2"), ("K4", "K3")]
        if protocol == "controlled":
            contrasts = [("K4", "K2")]
        for metric in METRICS:
            table = group.pivot(index="record_id", columns="condition", values=metric)
            for after, before in contrasts:
                if not {after, before} <= set(table):
                    continue
                if metric != "answer_relevancy" and ({after, before} & {"K1", "K3"}):
                    continue  # Jangan membandingkan metrik konteks dengan kondisi tanpa RAG.
                pair = table[[after, before]].dropna()
                if pair.empty:
                    continue
                if protocol == "controlled":
                    hashes = group.pivot(index="record_id", columns="condition", values="context_hash").loc[pair.index]
                    assert hashes[after].eq(hashes[before]).all(), "Konteks controlled berbeda!"
                delta = pair[after] - pair[before]
                rows.append({"protocol": protocol, "architecture": arch, "contrast": after+"-"+before,
                             "metric": metric, "n_pairs": len(pair), "mean_delta": delta.mean(),
                             "median_delta": delta.median()})
    return pd.DataFrame(rows, columns=["protocol", "architecture", "contrast", "metric", "n_pairs", "mean_delta", "median_delta"])

summary = summarize(scored, GROUP)
summary.to_csv(OUT / "system_metrics_summary.csv")
summarize(scored, GROUP + ["subtask_type"]).to_csv(OUT / "system_metrics_by_task.csv")
paired = paired_differences(scored)
paired.to_csv(OUT / "paired_differences.csv", index=False)
# JSON preserves list contexts; CSV detail intentionally contains scalar fields only.
scored.to_json(OUT / "system_scored.jsonl", orient="records", lines=True, force_ascii=False)
scalar_columns = [c for c in scored if not scored[c].map(lambda v: isinstance(v, (dict, list))).any()]
scored[scalar_columns].to_csv(OUT / "system_scored.csv", index=False)
coverage = predictions.groupby(GROUP + ["status"]).size().rename("n").reset_index()
coverage.to_csv(OUT / "inference_coverage.csv", index=False)
selected = pd.read_csv(OUT / "test_selected.csv", dtype=str)
expected = len(selected) * sum(
    1 for p in manifest["protocols"] for scenario in manifest["scenarios"]
    if p != "controlled" or scenario["use_rag"])
print("Prediksi tersimpan / diharapkan:", len(predictions), "/", expected)
print("PILOT — bukan hasil final skripsi" if manifest["pilot"] else "Full test configuration")
display(summary)
display(paired)

for metric in ["faithfulness", "answer_relevancy", "context_relevance"]:
    scored.groupby(GROUP + [metric + "_status"]).size().rename("n").to_csv(OUT / f"{metric}_coverage.csv")
write_json(OUT / "evaluation_scope.json", {"scope": "ragas_only_v1", "metrics": METRICS,
    "reference_answer_required": False, "run_ragas": RUN_RAGAS})


## 9. Simpan hasil

In [ ]:
import shutil
archive_path = shutil.make_archive(str(OUT.parent / RUN_NAME), "zip", root_dir=OUT.parent, base_dir=RUN_NAME)
print("Unduh seluruh folder/ZIP melalui panel Output Kaggle:", archive_path)
print("Tidak ada unggahan Hugging Face otomatis. Simpan output sebelum sesi Kaggle berakhir.")